# 07 · From visible harvests to bankable cash
## Action-opportunity feature investigation

**Question:** which worker opportunities disappear when we account for delivery time, crop decay, nonlinear sale prices and real shed capacity?

This notebook implements **35 numeric option descriptors and 14 state summaries**, plus eight audit-only probe counts. The output is a variable-length worker/option table, not a fixed worker-slot array. It retains all observed workers.

**This is not a leaderboard result.** The full agent callback has not been accepted, no new complete games are run, and the proposed action probes are not deployed to the frozen policy. The user-reported **3140.0** benchmark remains a research target.

Use the **Kaggriculture Manual (verified source)** kernel. Run this notebook only, not the full historical reproduction scripts.

In [1]:
from pathlib import Path
import importlib.util, json, subprocess, sys, signal, time
from IPython.display import display, Markdown, FileLink
import pandas as pd

BASE = Path.cwd().resolve()
if not (BASE / "run_next.py").is_file():
    BASE = Path.home() / "kaggriculture_action_features"
if not (BASE / "run_next.py").is_file():
    raise FileNotFoundError("Open notebook 07 from the extracted kaggriculture_action_features folder.")
OUT = BASE / "outputs"
print("LIVE WORKFLOW — no automatic synthetic or reference fallback")
print("Python:", sys.executable)
print("Package:", BASE)

def run_stage(stage):
    command = [sys.executable, str(BASE / "run_next.py"), stage]
    process = subprocess.Popen(command, cwd=BASE, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               start_new_session=True)
    try:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
        result = process.wait()
    except KeyboardInterrupt:
        import os
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=4)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise
    if result:
        raise RuntimeError(f"{stage} failed. Save this notebook and run the bundle command; do not retry unchanged.")


LIVE WORKFLOW — no automatic synthetic or reference fallback
Python: /home/sagemaker-user/projects/kaggriculture/.venv/bin/python
Package: /home/sagemaker-user/kaggriculture_action_features


## 1 · Reuse notebook 06 instead of guessing

The latest uploaded ZIP was the original input package, with no executed cells or result report. That does **not** imply the AWS run failed.

This cell checks the real local result. A verified pass is reused. If there has never been an execution, the existing 06 worker runs once with its 180-second cap. A previous failure/interruption stops here and preserves its evidence. No workspace reset, package installation, data redownload or Git write occurs.

In [2]:
run_stage("ensure06")
prerequisite = json.loads((OUT / "prerequisite.json").read_text())
display(pd.DataFrame([prerequisite]))

{"utc": "2026-09-11T22:35:50.938705+00:00", "stage": "NOTEBOOK06_GATE_PASSED", "reused": true}


,previous_report_sha256,previous_saved_observations,reused_existing_pass,source_commit,status,utc
0,5d432186a1e317362dee7a8164d4744a5d81155dd35a08...,161,True,7194116dfc92a8663139611233b5a221dad431a4,PASSED,2026-09-11T22:35:50.935039+00:00


## 2 · Hypothesis and representation

For a harvest target $t$ and worker $w$, the manual delivery cost is

$$C(w,t)=d(w,t)+1+d(t,\mathrm{shed})+1.$$

The two extra actions are HARVEST and DROP. The scenario has value only when it can finish in the remaining $719-\mathrm{step}$ decisions and the yield survives travel. Locked tiles do not block movement in the pinned interpreter.

The value is **incremental one-sided liquidation after existing shed stock**, holding the market fixed until arrival. It is not a forecast of actual future prices. DROP can discard goods before selling frees space. Existing carried goods, including their insertion order and competition for shed room, therefore matter.

The first-command alternatives are grouped explicitly. Two different targets that both require NORTH do not count as different immediate actions.

Source and full assumptions: `FEATURE_RESEARCH.md`. Metadata such as seed, episode outcome and the rival's private inventory never enters the extractor.

In [3]:
run_stage("run")
report = json.loads((OUT / "report.json").read_text())
fields = ["status", "source_episodes", "saved_observations", "candidate_rows",
          "numeric_candidate_descriptors", "aggregate_state_descriptors", "probe_diagnostic_columns",
          "primitive_parity_checks", "primitive_parity_passed", "new_complete_games",
          "full_callback_acceptance", "official_metric_effect_measured", "feature_value_proven"]
display(pd.DataFrame([{"check": key, "result": report[key]} for key in fields]))

{"utc": "2026-09-11T22:35:51.034674+00:00", "stage": "NOTEBOOK07_STARTED", "hard_limit_seconds": 180}
.........................................................................
----------------------------------------------------------------------
Ran 73 tests in 1.352s

OK
{"utc": "2026-09-11T22:35:53.698681+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-11T22:35:54.425975+00:00", "stage": "EPISODE_CHECKPOINTED", "episode": 1, "total": 7, "observations": 23, "candidate_rows": 307}
{"utc": "2026-09-11T22:35:55.126138+00:00", "stage": "EPISODE_CHECKPOINTED", "episode": 2, "total": 7, "observations": 23, "candidate_rows": 343}
{"utc": "2026-09-11T22:35:55.820126+00:00", "stage": "EPISODE_CHECKPOINTED", "episode": 3, "total": 7, "observations": 23, "candidate_rows": 307}
{"utc": "2026-09-11T22:35:56.512067+00:00", "stage": "EPISODE_CHECKPOINTED", "episode": 4, "total": 7, "observations": 23, "candidate_rows": 343}
{"utc": "2026-09-11T22:35:57.216362+00:00", "

,check,result
0,status,ACTION_FEATURE_AUDIT_PASSED
1,source_episodes,7
2,saved_observations,161
3,candidate_rows,2418
4,numeric_candidate_descriptors,35
5,aggregate_state_descriptors,14
6,probe_diagnostic_columns,8
7,primitive_parity_checks,30
8,primitive_parity_passed,True
9,new_complete_games,0


## 3 · Four ingredient-removal probes

All arms use the same deterministic selector and candidate identities; only one valuation ingredient is removed. PASS wins zero-value ties. These are **activation probes**, not outcome ablations or a deployed joint planner.

Interpretation: a zero first-command change means this intervention did not activate in that episode. A positive change identifies something to inspect—not evidence of improvement. Individual turns and workers are not treated as independent trials.

In [4]:
spec = importlib.util.spec_from_file_location("kaggriculture_action_visuals", BASE / "visualize.py")
visuals = importlib.util.module_from_spec(spec)
spec.loader.exec_module(visuals)
data = visuals.load_outputs(OUT)
charts = visuals.figures(data)
display(data["probes"][["seed", "seat", "source_arm", "arm", "evaluated_worker_states",
                       "changed_first_actions", "first_action_change_fraction"]])
charts[0].show(renderer="plotly_mimetype")

,seed,seat,source_arm,arm,evaluated_worker_states,changed_first_actions,first_action_change_fraction
0,1601,1,coordinated,full,86,0,0.000000
1,1601,1,coordinated,no_capacity,86,0,0.000000
2,1601,1,coordinated,no_deadline,86,0,0.000000
3,1601,1,coordinated,no_decay,86,0,0.000000
4,1601,1,coordinated,no_price_impact,86,0,0.000000
5,1601,0,sequential,full,86,0,0.000000
6,1601,0,sequential,no_capacity,86,0,0.000000
7,1601,0,sequential,no_deadline,86,0,0.000000
8,1601,0,sequential,no_decay,86,0,0.000000
9,1601,0,sequential,no_price_impact,86,0,0.000000


## 4 · Can the goods become cash before the clock stops?

Candidates to the left of zero slack cannot complete their manual delivery before the terminal boundary. Quantity-aware values remain conditional frozen-market estimates even for candidates to the right. The chart samples only for display when needed; extraction and exports retain every candidate.

In [5]:
charts[1].show(renderer="plotly_mimetype")

## 5 · Which workers have access to which opportunities?

The heatmap preserves original worker action indices. It displays one saved state, not an optimal allocation. Multiple workers may prefer the same harvest; those choices must be reconciled before a policy can use them. Do not add the cells into a joint income estimate.

In [6]:
display(Markdown("**Example metadata:** `" + json.dumps(data["example"]["metadata"], sort_keys=True) + "`"))
charts[2].show(renderer="plotly_mimetype")

**Example metadata:** `{"arm": "sequential", "episode_id": "6c5b293d1d6c0e2ac853", "opponent": "livestock_fertilizer", "seat": 1, "seed": 1602, "step": 699}`

## 6 · What headline price hides

Compare the same quantity/deadline/capacity scenario with and without marginal price impact. A current market quote multiplied by all units is not the proceeds of a sequential sale against a changing inventory curve. Neither axis predicts rival trading or future town demand.

In [7]:
charts[3].show(renderer="plotly_mimetype")

## 7 · Activation and representation coverage

Nonzero frequency is not feature importance. Constants in this final-day, latency-censored development slice are not grounds to reject a mechanism globally. No validation or holdout features are screened here.

In [8]:
charts[4].show(renderer="plotly_mimetype")
constants = data["registry"].query("distinct_values <= 1")
display(constants[["feature", "distinct_values", "nonzero_fraction"]])

,feature,distinct_values,nonzero_fraction
11,decayed_units_en_route,1,0.0
15,discarded_units,1,0.0
19,capacity_value_loss,1,0.0


## 8 · Runtime and primitive-mechanics parity

The primitive tests execute the pinned interpreter's unit actions and sale commits on **copied legal observations or labeled synthetic mechanics fixtures**, with the market frozen for that scenario. They do not simulate full new seasons. Their purpose is to check accounting and phase order.

Extractor latency excludes the rest of the game-playing callback. A fast plot here does not satisfy the existing 500 ms callback gate. Cached observations retain their original timings.

In [9]:
charts[5].show(renderer="plotly_mimetype")
parity = pd.read_csv(OUT / "mechanics_parity.csv")
display(parity)
display(pd.DataFrame([report["extractor_ms"]]))

,source,case,episode_id,step,candidate_id,accepted_units,measured_incremental_frozen_revenue,primitive_action_steps,passed
0,synthetic_mechanics,0.0,NaN,NaN,w0:deliver,3,70.0,1,True
1,synthetic_mechanics,1.0,NaN,NaN,w0:harvest:4:4,4,93.0,2,True
2,synthetic_mechanics,2.0,NaN,NaN,w0:harvest:2:0,5,116.0,10,True
3,synthetic_mechanics,3.0,NaN,NaN,w0:deliver,6,547.0,1,True
4,synthetic_mechanics,3.0,NaN,NaN,w0:harvest:4:4,6,145.0,2,True
5,synthetic_mechanics,4.0,NaN,NaN,w0:deliver,6,805.0,1,True
6,synthetic_mechanics,4.0,NaN,NaN,w0:harvest:4:4,7,439.0,2,True
7,synthetic_mechanics,5.0,NaN,NaN,w0:harvest:4:4,4,93.0,2,True
8,synthetic_mechanics,5.0,NaN,NaN,w1:harvest:4:4,4,93.0,2,True
9,saved_observation,NaN,35cba1b6eb288052fa45,696.0,w0:harvest:1:2,2,164.0,12,True


,max,median,p95
0,3.132844,1.359625,2.787731


## 9 · Persist the evidence and decide

Inspect action changes by episode and seed/opponent block. Before spending on new games, select one activated, mechanically sound family for integration while holding hiring, market rules and the solver fixed. Full-callback acceptance and fresh grouped paired outcomes remain required. Feature engineering remains open.

In [10]:
dashboard = visuals.export_dashboard(data, charts, OUT / "action_opportunities.html")
blocks = pd.read_csv(OUT / "probe_by_block.csv")
display(blocks)
print("Standalone interactive dashboard:", dashboard)
print("Official metric improvement measured:", report["official_metric_effect_measured"])
print("Full callback acceptance:", report["full_callback_acceptance"])
print("No new complete games or model fits were run.")
display(FileLink("outputs/action_opportunities.html"))

,seed,opponent,arm,observed_episodes,mean_episode_action_change_fraction
0,1601,livestock_fertilizer,full,4,0.000000
1,1601,livestock_fertilizer,no_capacity,4,0.000000
2,1601,livestock_fertilizer,no_deadline,4,0.000000
3,1601,livestock_fertilizer,no_decay,4,0.000000
4,1601,livestock_fertilizer,no_price_impact,4,0.000000
5,1602,livestock_fertilizer,full,3,0.000000
6,1602,livestock_fertilizer,no_capacity,3,0.000000
7,1602,livestock_fertilizer,no_deadline,3,0.073643
8,1602,livestock_fertilizer,no_decay,3,0.000000
9,1602,livestock_fertilizer,no_price_impact,3,0.000000


Standalone interactive dashboard: /home/sagemaker-user/kaggriculture_action_features/outputs/action_opportunities.html
Official metric improvement measured: False
Full callback acceptance: NOT_RUN
No new complete games or model fits were run.


/home/sagemaker-user/kaggriculture_action_features/outputs/action_opportunities.html

## 10 · Save and return the **results** ZIP

Press **Ctrl+S** now. In a terminal, run the command in `START_HERE.md` that ends with `run_next.py bundle`. Download and upload **`kaggriculture_action_feature_results.zip`**, not the original input package.

If an earlier cell failed, the same bundle command includes the failure details. Stop the JupyterLab application afterward; do not delete the space.